# 电商用户画像分析报告（优化版）

## 优化内容
- 分类规则改为JSON配置文件驱动
- PySpark性能优化：广播变量 + 数据重分区
- 偏好品类计算：最近30天购买频次加权排序
- 增加A/B测试对比功能

In [ ]:
import json
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StringType, DoubleType, IntegerType
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from datetime import datetime, timedelta

%matplotlib inline
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
spark = SparkSession.builder \
    .appName("电商用户画像分析_优化版") \
    .config("spark.sql.warehouse.dir", "/user/hive/warehouse") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .enableHiveSupport() \
    .getOrCreate()

print("PySpark版本:", spark.version)

## 1. 加载配置文件

In [ ]:
with open('classification_rules.json', 'r', encoding='utf-8') as f:
    config = json.load(f)

print("配置加载成功!")
print(f"分析日期: {config['analysis_config']['analysis_date']}")
print(f"偏好计算天数: {config['analysis_config']['preference_days']}天")
print(f"分区数: {config['analysis_config']['num_partitions']}")
print(f"\n用户分类规则:")
for seg in config['segments']:
    print(f"  - {seg['name']}: {seg['conditions']}")

## 2. 数据生成与加载（含性能优化）

In [ ]:
np.random.seed(42)
num_users = 10000
user_ids = [f"user_{i:06d}" for i in range(1, num_users + 1)]
categories = ['电子产品', '服装鞋帽', '食品饮料', '家居用品', '美妆护肤', '母婴用品', '运动户外']
analysis_date = datetime.strptime(config['analysis_config']['analysis_date'], '%Y-%m-%d')
num_partitions = config['analysis_config']['num_partitions']

In [ ]:
def generate_orders_data():
    orders = []
    base_date = analysis_date - timedelta(days=365)
    
    for user_id in user_ids:
        user_type = np.random.choice(['high', 'growth', 'sleep', 'churn'], p=[0.15, 0.35, 0.25, 0.25])
        
        if user_type == 'high':
            num_orders = np.random.randint(20, 100)
            avg_amount = np.random.uniform(500, 2000)
            days_back = np.random.randint(0, 30)
        elif user_type == 'growth':
            num_orders = np.random.randint(5, 30)
            avg_amount = np.random.uniform(200, 800)
            days_back = np.random.randint(0, 60)
        elif user_type == 'sleep':
            num_orders = np.random.randint(3, 15)
            avg_amount = np.random.uniform(100, 500)
            days_back = np.random.randint(60, 180)
        else:
            num_orders = np.random.randint(1, 5)
            avg_amount = np.random.uniform(50, 300)
            days_back = np.random.randint(180, 365)
        
        for _ in range(num_orders):
            order_date = analysis_date - timedelta(days=days_back - np.random.randint(0, days_back + 1))
            category = np.random.choice(categories)
            amount = np.random.uniform(avg_amount * 0.5, avg_amount * 1.5)
            
            orders.append({
                'user_id': user_id,
                'order_id': f"order_{np.random.randint(1000000, 9999999)}",
                'order_date': order_date.strftime('%Y-%m-%d'),
                'category': category,
                'amount': round(amount, 2),
                'quantity': np.random.randint(1, 5)
            })
    
    return spark.createDataFrame(pd.DataFrame(orders))

orders_df = generate_orders_data().repartition(num_partitions, 'user_id').cache()
print(f"订单数据: {orders_df.count()} 条, 分区数: {orders_df.rdd.getNumPartitions()}")
orders_df.show(5)

In [ ]:
def generate_browse_data():
    browses = []
    
    for user_id in user_ids:
        user_type = np.random.choice(['high', 'growth', 'sleep', 'churn'], p=[0.15, 0.35, 0.25, 0.25])
        
        if user_type == 'high':
            num_browses = np.random.randint(100, 500)
            days_back = np.random.randint(0, 15)
        elif user_type == 'growth':
            num_browses = np.random.randint(30, 150)
            days_back = np.random.randint(0, 45)
        elif user_type == 'sleep':
            num_browses = np.random.randint(10, 50)
            days_back = np.random.randint(45, 150)
        else:
            num_browses = np.random.randint(1, 20)
            days_back = np.random.randint(150, 365)
        
        for _ in range(num_browses):
            browse_date = analysis_date - timedelta(days=days_back - np.random.randint(0, days_back + 1))
            category = np.random.choice(categories)
            duration = np.random.randint(10, 300)
            
            browses.append({
                'user_id': user_id,
                'browse_id': f"browse_{np.random.randint(1000000, 9999999)}",
                'browse_date': browse_date.strftime('%Y-%m-%d'),
                'category': category,
                'duration': duration
            })
    
    return spark.createDataFrame(pd.DataFrame(browses))

browse_df = generate_browse_data().repartition(num_partitions, 'user_id').cache()
print(f"浏览数据: {browse_df.count()} 条, 分区数: {browse_df.rdd.getNumPartitions()}")
browse_df.show(5)

In [ ]:
def generate_reviews_data():
    reviews = []
    
    for user_id in user_ids:
        user_type = np.random.choice(['high', 'growth', 'sleep', 'churn'], p=[0.15, 0.35, 0.25, 0.25])
        
        if user_type == 'high':
            num_reviews = np.random.randint(10, 50)
            avg_rating = np.random.uniform(3.5, 5.0)
        elif user_type == 'growth':
            num_reviews = np.random.randint(3, 20)
            avg_rating = np.random.uniform(3.0, 4.5)
        elif user_type == 'sleep':
            num_reviews = np.random.randint(1, 8)
            avg_rating = np.random.uniform(2.5, 4.0)
        else:
            num_reviews = np.random.randint(0, 3)
            avg_rating = np.random.uniform(2.0, 3.5)
        
        for _ in range(num_reviews):
            review_date = analysis_date - timedelta(days=np.random.randint(0, 365))
            rating = min(5, max(1, round(np.random.normal(avg_rating, 0.5), 1)))
            
            reviews.append({
                'user_id': user_id,
                'review_id': f"review_{np.random.randint(1000000, 9999999)}",
                'review_date': review_date.strftime('%Y-%m-%d'),
                'rating': rating,
                'has_image': np.random.choice([True, False], p=[0.3, 0.7])
            })
    
    return spark.createDataFrame(pd.DataFrame(reviews))

reviews_df = generate_reviews_data().repartition(num_partitions, 'user_id').cache()
print(f"评论数据: {reviews_df.count()} 条, 分区数: {reviews_df.rdd.getNumPartitions()}")
reviews_df.show(5)

## 3. 使用广播变量优化小表关联

In [ ]:
category_info = pd.DataFrame({
    'category': categories,
    'category_id': range(1, len(categories) + 1),
    'avg_price_level': ['高', '中', '低', '中', '中高', '中', '中高']
})

category_info_df = spark.createDataFrame(category_info)
broadcast_categories = F.broadcast(category_info_df)

print("品类维度表（已广播）:")
broadcast_categories.show()

## 4. 用户属性计算（优化版）

In [ ]:
weights = config['weights']
analysis_date_str = config['analysis_config']['analysis_date']

order_activity = orders_df.groupBy('user_id').agg(
    F.count('order_id').alias('order_count'),
    F.sum('amount').alias('total_amount'),
    F.max('order_date').alias('last_order_date'),
    F.datediff(F.lit(analysis_date_str), F.max('order_date')).alias('days_since_last_order')
)

browse_activity = browse_df.groupBy('user_id').agg(
    F.count('browse_id').alias('browse_count'),
    F.sum('duration').alias('total_browse_duration'),
    F.max('browse_date').alias('last_browse_date'),
    F.datediff(F.lit(analysis_date_str), F.max('browse_date')).alias('days_since_last_browse')
)

review_activity = reviews_df.groupBy('user_id').agg(
    F.count('review_id').alias('review_count'),
    F.avg('rating').alias('avg_rating')
)

user_activity = order_activity.join(browse_activity, 'user_id', 'outer') \
    .join(review_activity, 'user_id', 'outer') \
    .fillna(0)

user_activity = user_activity.withColumn(
    'activity_score',
    F.log1p(
        F.col('order_count') * weights['order_count_weight'] +
        F.col('browse_count') * weights['browse_count_weight'] +
        F.col('review_count') * weights['review_count_weight']
    )
).withColumn(
    'recency_score',
    F.when((F.col('days_since_last_order') == 0) & (F.col('days_since_last_browse') > 0), F.col('days_since_last_browse'))
     .when((F.col('days_since_last_browse') == 0) & (F.col('days_since_last_order') > 0), F.col('days_since_last_order'))
     .when((F.col('days_since_last_order') == 0) & (F.col('days_since_last_browse') == 0), 365)
     .otherwise(F.least(F.col('days_since_last_order'), F.col('days_since_last_browse')))
)

print("用户活跃度数据:")
user_activity.select('user_id', 'order_count', 'browse_count', 'activity_score', 'recency_score').show(10)

In [ ]:
order_stats = orders_df.groupBy('user_id').agg(
    F.sum('amount').alias('total_spend'),
    F.avg('amount').alias('avg_order_value'),
    F.countDistinct(F.date_format('order_date', 'yyyy-MM')).alias('active_months'),
    F.sum('quantity').alias('total_quantity')
)

order_stats = order_stats.withColumn(
    'consumption_score',
    F.log1p(F.col('total_spend')) * F.col('active_months') / 12
)

print("用户消费能力数据:")
order_stats.select('user_id', 'total_spend', 'avg_order_value', 'active_months', 'consumption_score').show(10)

### 4.1 修复偏好品类计算：最近30天购买频次加权

In [ ]:
preference_days = config['analysis_config']['preference_days']
print(f"计算最近{preference_days}天的偏好品类...")

cutoff_date = analysis_date - timedelta(days=preference_days)
cutoff_date_str = cutoff_date.strftime('%Y-%m-%d')

recent_orders = orders_df.filter(F.col('order_date') >= cutoff_date_str)

category_pref = recent_orders.groupBy('user_id', 'category').agg(
    F.sum('amount').alias('category_spend'),
    F.count('order_id').alias('category_orders'),
    F.sum('quantity').alias('category_quantity')
)

category_pref = category_pref.withColumn(
    'preference_score',
    F.col('category_orders') * 0.5 + F.log1p(F.col('category_spend')) * 0.3 + F.col('category_quantity') * 0.2
)

window_spec = Window.partitionBy('user_id').orderBy(F.desc('preference_score'))
user_preference = category_pref.withColumn('rank', F.rank().over(window_spec)) \
    .filter(F.col('rank') == 1) \
    .select('user_id', 
            F.col('category').alias('preferred_category'), 
            'category_spend', 
            'category_orders',
            'preference_score')

print(f"最近{preference_days}天用户偏好品类:")
user_preference.show(10)

In [ ]:
category_avg_price = orders_df.groupBy('category').agg(
    F.avg('amount').alias('category_avg_price')
)

broadcast_category_price = F.broadcast(category_avg_price)

user_price_behavior = orders_df.join(broadcast_category_price, 'category') \
    .withColumn('price_ratio', F.col('amount') / F.col('category_avg_price')) \
    .groupBy('user_id').agg(
        F.avg('price_ratio').alias('avg_price_ratio'),
        F.stddev('price_ratio').alias('price_ratio_std')
    )

price_levels = config['price_sensitivity']['levels']
price_sensitivity_expr = F.when(F.col('avg_price_ratio') < price_levels[0]['max_ratio'], price_levels[0]['score'])
for level in price_levels[1:-1]:
    price_sensitivity_expr = price_sensitivity_expr.when(
        F.col('avg_price_ratio') < level['max_ratio'], level['score']
    )
price_sensitivity_expr = price_sensitivity_expr.otherwise(price_levels[-1]['score'])

user_price_behavior = user_price_behavior.withColumn('price_sensitivity', price_sensitivity_expr)

print("用户价格敏感度:")
user_price_behavior.select('user_id', 'avg_price_ratio', 'price_sensitivity').show(10)

## 5. 基于配置的用户分群

In [ ]:
user_features = user_activity.join(order_stats, 'user_id', 'outer') \
    .join(user_preference, 'user_id', 'outer') \
    .join(user_price_behavior, 'user_id', 'outer') \
    .fillna(0) \
    .repartition(num_partitions, 'user_id')

print(f"总用户数: {user_features.count()}")
print(f"特征数据分区数: {user_features.rdd.getNumPartitions()}")

In [ ]:
def create_segmentation_udf(segments_config):
    def classify_user(recency, activity, consumption):
        recency_val = recency if recency and recency > 0 else 365
        activity_val = activity if activity else 0
        consumption_val = consumption if consumption else 0
        
        sorted_segments = sorted(segments_config, key=lambda x: x['priority'])
        
        for seg in sorted_segments:
            cond = seg['conditions']
            match = True
            
            if 'recency_max' in cond and recency_val > cond['recency_max']:
                match = False
            if 'recency_min' in cond and recency_val < cond['recency_min']:
                match = False
            if 'consumption_min' in cond and consumption_val < cond['consumption_min']:
                match = False
            if 'activity_min' in cond and activity_val < cond['activity_min']:
                match = False
            
            if match:
                return seg['name']
        
        return '未知用户'
    
    return F.udf(classify_user, StringType())

In [ ]:
segment_udf = create_segmentation_udf(config['segments'])

user_segments = user_features.withColumn(
    'segment',
    segment_udf(F.col('recency_score'), F.col('activity_score'), F.col('consumption_score'))
).cache()

print("用户分群结果（基于配置文件）:")
segment_counts = user_segments.groupBy('segment').count().orderBy('count', ascending=False)
segment_counts.show()

## 6. A/B测试对比功能

In [ ]:
ab_config = config['ab_test']

if ab_config['enabled']:
    print(f"执行A/B测试: {ab_config['test_name']}")
    print(f"对照组: {ab_config['control_group']['name']}")
    print(f"实验组: {ab_config['experimental_group']['name']}")
    
    control_segments = ab_config['control_group']['segments']
    experimental_segments = config['segments'] if ab_config['experimental_group'].get('use_main_config', False) else ab_config['experimental_group']['segments']
    
    control_udf = create_segmentation_udf(control_segments)
    experimental_udf = create_segmentation_udf(experimental_segments)
    
    ab_results = user_features.withColumn(
        'control_segment',
        control_udf(F.col('recency_score'), F.col('activity_score'), F.col('consumption_score'))
    ).withColumn(
        'experimental_segment',
        experimental_udf(F.col('recency_score'), F.col('activity_score'), F.col('consumption_score'))
    ).cache()
    
    print("\nA/B测试数据生成完成!")

In [ ]:
if ab_config['enabled']:
    control_counts = ab_results.groupBy('control_segment').count() \
        .withColumnRenamed('control_segment', 'segment') \
        .withColumnRenamed('count', 'control_count')
    
    experimental_counts = ab_results.groupBy('experimental_segment').count() \
        .withColumnRenamed('experimental_segment', 'segment') \
        .withColumnRenamed('count', 'experimental_count')
    
    ab_comparison = control_counts.join(experimental_counts, 'segment', 'outer').fillna(0)
    ab_comparison = ab_comparison.withColumn(
        'diff_pct',
        F.round((F.col('experimental_count') - F.col('control_count')) / F.col('control_count') * 100, 2)
    ).toPandas()
    
    segment_order = ['高价值用户', '成长用户', '沉睡用户', '流失用户', '未知用户']
    ab_comparison['segment'] = pd.Categorical(ab_comparison['segment'], categories=segment_order, ordered=True)
    ab_comparison = ab_comparison.sort_values('segment').reset_index(drop=True)
    
    print("A/B测试对比结果:")
    print(f"{'='*60}")
    print(f"{'用户群':<12} {'对照组':>10} {'实验组':>10} {'变化%':>10}")
    print(f"{'='*60}")
    for _, row in ab_comparison.iterrows():
        diff_str = f"{row['diff_pct']:+.2f}%"
        print(f"{row['segment']:<12} {row['control_count']:>10} {row['experimental_count']:>10} {diff_str:>10}")
    print(f"{'='*60}")

In [ ]:
if ab_config['enabled']:
    confusion_matrix = ab_results.crosstab('control_segment', 'experimental_segment').toPandas()
    confusion_matrix = confusion_matrix.set_index('control_segment_experimental_segment')
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(confusion_matrix, annot=True, fmt='d', cmap='YlOrRd', cbar_kws={'label': '用户数'})
    plt.title('A/B测试用户迁移矩阵', fontsize=14, fontweight='bold')
    plt.xlabel(f"实验组 ({ab_config['experimental_group']['name']})", fontsize=12)
    plt.ylabel(f"对照组 ({ab_config['control_group']['name']})", fontsize=12)
    plt.tight_layout()
    plt.show()

## 7. 分析结果可视化

In [ ]:
segment_counts_pd = segment_counts.toPandas()
segment_counts_pd = segment_counts_pd.sort_values('count', ascending=False)

plt.figure(figsize=(10, 6))
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4']
wedges, texts, autotexts = plt.pie(
    segment_counts_pd['count'],
    labels=segment_counts_pd['segment'],
    autopct='%1.1f%%',
    colors=colors,
    startangle=90
)
plt.title('用户分群占比分布', fontsize=16, fontweight='bold')
plt.axis('equal')
plt.show()

print("\n用户分群统计表:")
print(segment_counts_pd.to_string(index=False))

In [ ]:
segment_analysis = user_segments.groupBy('segment').agg(
    F.count('user_id').alias('user_count'),
    F.round(F.avg('total_spend'), 2).alias('avg_total_spend'),
    F.round(F.avg('order_count'), 2).alias('avg_order_count'),
    F.round(F.avg('activity_score'), 2).alias('avg_activity_score'),
    F.round(F.avg('recency_score'), 1).alias('avg_recency_days'),
    F.round(F.avg('price_sensitivity'), 2).alias('avg_price_sensitivity')
).toPandas()

segment_order = ['高价值用户', '成长用户', '沉睡用户', '流失用户']
segment_analysis['segment'] = pd.Categorical(segment_analysis['segment'], categories=segment_order, ordered=True)
segment_analysis = segment_analysis.sort_values('segment').reset_index(drop=True)

print("用户分群特征分析:")
display(segment_analysis)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('用户分群特征对比', fontsize=18, fontweight='bold')

sns.barplot(data=segment_analysis, x='segment', y='avg_total_spend', ax=axes[0,0], palette=colors)
axes[0,0].set_title('平均消费金额', fontsize=12)
axes[0,0].set_xlabel('')
axes[0,0].tick_params(axis='x', rotation=15)

sns.barplot(data=segment_analysis, x='segment', y='avg_order_count', ax=axes[0,1], palette=colors)
axes[0,1].set_title('平均订单数量', fontsize=12)
axes[0,1].set_xlabel('')
axes[0,1].tick_params(axis='x', rotation=15)

sns.barplot(data=segment_analysis, x='segment', y='avg_activity_score', ax=axes[1,0], palette=colors)
axes[1,0].set_title('平均活跃度评分', fontsize=12)
axes[1,0].set_xlabel('')
axes[1,0].tick_params(axis='x', rotation=15)

sns.barplot(data=segment_analysis, x='segment', y='avg_recency_days', ax=axes[1,1], palette=colors)
axes[1,1].set_title('平均最近活跃天数', fontsize=12)
axes[1,1].set_xlabel('')
axes[1,1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

In [ ]:
category_dist = user_segments.groupBy('segment', 'preferred_category').count() \
    .filter(F.col('preferred_category') != 0) \
    .toPandas()

if len(category_dist) > 0:
    plt.figure(figsize=(14, 8))
    pivot_df = category_dist.pivot(index='segment', columns='preferred_category', values='count').fillna(0)
    pivot_df_pct = pivot_df.div(pivot_df.sum(axis=1), axis=0) * 100

    pivot_df_pct.plot(kind='bar', stacked=True, figsize=(14, 8), colormap='Set3')
    plt.title(f'各用户群偏好品类分布（最近{preference_days}天加权）', fontsize=16, fontweight='bold')
    plt.xlabel('用户分群', fontsize=12)
    plt.ylabel('占比 (%)', fontsize=12)
    plt.legend(title='品类', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.show()
else:
    print("暂无最近偏好数据")

## 8. 总结报告

In [ ]:
print("=" * 80)
print("\t\t电商用户画像分析报告（优化版）")
print("=" * 80)
print(f"\n分析日期: {analysis_date_str}")
print(f"分析用户数: {user_segments.count()}")
print(f"偏好计算窗口: 最近{preference_days}天")
print(f"数据分区数: {num_partitions}")
print()

for _, row in segment_analysis.iterrows():
    print(f"\n【{row['segment']}】")
    print(f"  用户数: {row['user_count']} 人 ({row['user_count']/user_segments.count()*100:.1f}%)")
    print(f"  平均消费: ¥{row['avg_total_spend']:.2f}")
    print(f"  平均订单数: {row['avg_order_count']:.1f} 单")
    print(f"  活跃度评分: {row['avg_activity_score']:.2f}")
    print(f"  最近活跃: {row['avg_recency_days']:.0f} 天前")
    print(f"  价格敏感度: {row['avg_price_sensitivity']:.1f}/5 (1=高端, 5=敏感)")

print("\n" + "=" * 80)
print("\n配置驱动特性:")
print("- ✓ 分类规则可通过JSON配置动态调整")
print("- ✓ 权重参数可配置")
print("- ✓ 价格敏感度等级可配置")

print("\n性能优化特性:")
print(f"- ✓ 数据重分区（{num_partitions}分区，按user_id哈希）")
print("- ✓ 广播变量优化小表关联")
print("- ✓ 数据Cache缓存")
print("- ✓ 自适应查询执行（AQE）")

if ab_config['enabled']:
    print(f"\nA/B测试功能: ✓ 已启用 ({ab_config['test_name']})")

print("\n运营建议:")
print("- 高价值用户: 提供VIP专属服务，定期推送高端新品")
print("- 成长用户: 设计成长激励体系，推荐高性价比商品")
print("- 沉睡用户: 发送召回优惠券，推送个性化推荐")
print("- 流失用户: 调研流失原因，设计回归大礼包")
print("\n" + "=" * 80)

In [ ]:
spark.stop()
print("分析完成，Spark会话已关闭")